# Stacking and Blending Ensembles

---

# Stacking

Stacking is an ensemble technique similar to Voting Ensemble.

Like voting:

- multiple base models are used
- base models are usually different algorithms

Examples:

- Decision Tree
- Logistic Regression
- SVM
- KNN

---

## Working

First, all base models make predictions.

These predictions are then used as input features for another model called:

## Meta Model

The meta model learns:

- which base models are reliable
- how much importance to give each model

Thus final prediction is produced by the meta model.


---

# Difference from Bagging and Boosting

## Stacking

- Uses different base models
- Requires a meta model
- Learns how to combine predictions

---

## Bagging

- Usually uses same type of model
- No meta model
- Combines predictions through voting or averaging

---

## Boosting

- Uses same weak learner repeatedly
- Models are trained sequentially
- No meta model

---

# Problem with Stacking

A major problem is:

## Overfitting

If the same data is used:

- to train base models
- and to train meta model

then the meta model may simply memorize predictions.

---

# Solution 1: Holdout Method (Blending)

Blending is a simplified version of stacking.

The training data is divided into:

- training split
- validation split

---

## Step 1

Train base models on:

$$
\text{Training Split}
$$

---

## Step 2

Use trained base models to generate predictions on:

$$
\text{Validation Split}
$$

---

## Step 3

These predictions become inputs for the meta model.

The meta model is trained on:

$$
\text{Validation Split}
$$

instead of training data.

This reduces overfitting.

---

## Final Prediction

For new data:

1. Base models generate predictions
2. Predictions are sent to meta model
3. Meta model produces final output

---

# Solution 2: K-Fold Method (True Stacking)

![](./images2/img9.png)

Instead of one train-validation split, we use:

## K-Fold Cross Validation

---

## Step 1

Choose:

$$
K
$$

and divide data into:

$$
K
$$

equal parts.

---

## Step 2

Train base models on:

$$
K-1
$$

parts

and predict on remaining part.

---

## Step 3

Repeat using different fold combinations until every sample has been predicted once.

Thus we obtain predictions for the entire dataset.

---

## Step 4

Collect predictions from all base models.


---

## Step 5

Train the meta model on this new dataset.

The meta model learns how to combine base model predictions.

---

## Step 6

After meta model training:

- retrain all base models on the entire original dataset

This ensures maximum learning before deployment.

---

# Summary

## Blending

- Uses holdout validation set
- Easier to implement
- Faster
- Slightly less accurate

---

## Stacking

- Uses K-Fold Cross Validation
- Uses entire dataset efficiently
- Less overfitting
- More computationally expensive

---

# Key Idea

Stacking does not simply vote.

Instead, it learns how to combine predictions using a meta model, making it one of the most powerful ensemble techniques.



# Multi Layer Stacking

Multi Layer Stacking is an extension of stacking where multiple layers of models are used.

Instead of having:

- one layer of base models
- one meta model

we stack several layers of models on top of each other.

---

## Architecture

![](./images2/img10.png)

---

## Working

### Layer 1

Multiple base models are trained on the dataset.

Examples:

- Decision Tree
- Random Forest
- SVM
- Logistic Regression
- KNN

These models generate predictions.

---

### Layer 2

Predictions from Layer 1 become inputs to another set of models.

These models learn patterns in the predictions generated by Layer 1.

Again, predictions are generated.

---

### Layer 3 (Meta Layer)

Predictions from Layer 2 are fed into the final meta model.

The meta model produces the final prediction.

---

## Why Use Multiple Layers?

Each layer learns:

- strengths of previous models
- weaknesses of previous models
- interactions between model predictions

This can improve predictive performance for complex problems.

---

# Multi Layer Blending

When using the blending approach, the dataset is typically divided into three parts.

## Split 1: Training Set

Used to train base models.

---

## Split 2: Validation Set

Used to generate predictions from base models.

These predictions are used to train higher-layer models or meta models.

---

## Split 3: Test Set

Kept completely unseen during training.

Used only for final evaluation of the ensemble.

---




In [14]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv("./dataset/heart.csv")

In [16]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [17]:
X = df.drop(columns=["target"])
y = df["target"]

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

In [20]:
estimators = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=10)),
    ('gbdt', GradientBoostingClassifier())
]

In [21]:
from sklearn.ensemble import StackingClassifier

clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv = 10
)

In [22]:
clf.fit(X_train,y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('rf', ...), ('knn', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegression()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",10
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples re

In [23]:
y_pred = clf.predict(X_test)

In [24]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.8852459016393442